# 04 · The LangGraph investigation: workflow, bounded agent, human decision

**The system-design point:** the required checks are a **fixed workflow** (predictable, auditable, cheap,
testable). Agent autonomy is used in exactly **one** place, the deep dive, where the next step genuinely depends
on what was found. Money only moves through a **human** decision, via `interrupt()`.

This notebook uses the **real** Gemini model for extraction, AI review, deep dive and report. It runs against
the **scratch database** (`trustagent_test`), which it resets first, so the demo data is never touched.

In [1]:
import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import create_engine, select, text
from sqlalchemy.orm import sessionmaker

from trustagent.config import get_settings
from trustagent.db.models import AuditLogRow, InvestigationRow
from trustagent.db.seed import seed
from trustagent.domain import HumanAction
from trustagent.rag.embeddings import GeminiEmbedder
from trustagent.rag.ingest import ingest_all
from trustagent.workflow.identity import DEMO_USERS
from trustagent.workflow.intake import intake_document
from trustagent.workflow.service import InvestigationService, default_deps

settings = get_settings()
pd.set_option("display.max_colwidth", 110)
print(
    f"Model: {settings.llm_provider} / {settings.llm_model} (real API calls), paced at "
    f"{settings.llm_requests_per_minute[settings.llm_provider]:.0f} requests/min"
)

# Scratch database: reset, then seed the reference suppliers, history and policies.
engine = create_engine(settings.test_database_url)
with engine.begin() as c:
    c.execute(
        text(
            "TRUNCATE audit_log, evidence, investigations, invoices, transactions, suppliers, policies, "
            "document_chunks "
            "RESTART IDENTITY CASCADE"
        )
    )
    for t in ("checkpoints", "checkpoint_blobs", "checkpoint_writes"):
        if c.execute(text("select to_regclass(:t)"), {"t": t}).scalar():
            c.execute(text(f"TRUNCATE {t}"))
sf = sessionmaker(bind=engine, expire_on_commit=False)
with sf() as s, s.begin():
    seed(s)

# Contracts and the policy manual, for hybrid retrieval (real Gemini embeddings).
with sf() as s, s.begin():
    print("ingested:", ingest_all(s, GeminiEmbedder()))

service = InvestigationService.create(settings.test_database_url, deps=default_deps(sf))
thandi, sipho, lerato = DEMO_USERS["thandi"], DEMO_USERS["sipho"], DEMO_USERS["lerato"]

Model: gemini / gemini-3.5-flash-lite (real API calls), paced at 14 requests/min


ingested: {'ABC-2026-017': 11, 'MC-2025-004': 11, 'MC-2024-001': 10, 'DPC-2025-009': 11, 'FIN-POL-01': 33}


## 1. The graph

Built by `trustagent.graph.builder.build_graph`. `retrieve_context` runs hybrid RAG over the supplier's contract
and the policy manual; `contract_check` is pure code comparing invoice prices with the contract rates the AI
mapped. Two conditional branches: the **deep dive** (only when the provisional result is MEDIUM or ambiguous)
and the **decision loop** (back to `human_decision` while the case is still open).

In [2]:
display(Markdown("```mermaid\n" + service.graph.get_graph().draw_mermaid() + "\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_context(load_context)
	rule_checks(rule_checks)
	retrieve_context(retrieve_context)
	ai_review(ai_review)
	contract_check(contract_check)
	provisional_score(provisional_score)
	deep_dive(deep_dive)
	score(score)
	report(report)
	human_decision(human_decision)
	execute_action(execute_action)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_context;
	ai_review --> contract_check;
	contract_check --> provisional_score;
	deep_dive --> score;
	execute_action -. &nbsp;end&nbsp; .-> __end__;
	execute_action -.-> human_decision;
	human_decision --> execute_action;
	load_context --> rule_checks;
	provisional_score -.-> deep_dive;
	provisional_score -.-> score;
	report --> human_decision;
	retrieve_context --> ai_review;
	rule_checks --> retrieve_context;
	score --> report;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## 2. Upload a business-email-compromise invoice

INV-2003 claims Digital Print Co's Absa account is "temporarily frozen" and asks for payment to a new Capitec
account, from a look-alike email domain, "payment required today", "do not call our office".
Intake extracts it (real model), matches the supplier and opens a `PENDING` case.

In [3]:
pdf = settings.invoices_dir / "INV-2003-digital-print-co-BANK-CHANGE.pdf"
with sf() as s, s.begin():
    intake = intake_document(s, pdf.name, pdf.read_bytes(), submitted_by="Thandi Nkosi")
case_id = intake.investigation_id
print(
    case_id,
    "|",
    intake.invoice.invoice_number,
    "|",
    intake.supplier_match,
    intake.supplier_id,
    "| amount",
    intake.invoice.amount,
    "| account",
    intake.invoice.bank_account,
)

CASE-1788FFAB | INV-2003 | MATCHED_EXISTING SUP-003 | amount 43585.00 | account ****5570


## 3. Run the investigation and watch the stream

`service.start` claims the case **atomically** (`UPDATE … WHERE status='PENDING'`) and streams events.
Note the order: every **evidence** item comes **before** the **risk** score, then the recommendation, then the
run **pauses** for a human.

In [4]:
def show(events):
    for e in events:
        kind = e["event"]
        if kind == "activity":
            print(f"  · {e['action']}: {e['detail'][:150]}")
        elif kind == "evidence" and e["type"] != "CONFIRMED_MATCH":
            print(f"  ▲ EVIDENCE {e['type']} [{e['source']}, +{e['weight']}] {e['description'][:120]}")
        elif kind == "risk":
            print(f"  ■ RISK {e['score']}/100 {e['level']}")
        elif kind == "recommendation":
            print(f"  ► RECOMMENDATION {e['recommended_action']}: {e['recommendation'][:220]}")
        elif kind == "awaiting_decision":
            print(f"  ⏸ PAUSED for a human decision. Options: {', '.join(e['options'])}")
        elif kind in ("action_result", "complete", "error"):
            print(f"  {kind.upper()}: {e}")


show(service.start(case_id, thandi))

  · Invoice and supplier loaded: INV-2003 from Digital Print Co (Pty) Ltd


  · Rule checks completed: 3 finding(s), 1 check(s) passed


  · Policies and contracts retrieved: contract on file; 8 passage(s): DPC-2025-009 (contract), FIN-POL-01 (policy)


  · AI document review completed: 2 observation(s)
  · Contract check completed: 0 price deviation(s), 3 line(s) at the contract rate


  · Deep dive started: Score 85 is within 5 points of the 80 boundary.


  · Deep dive completed: Tools used: supplier_payment_history, similar_invoices, open_cases_for_supplier. Investigation into Digital Print Co's invoice revealed no additional 
  ▲ EVIDENCE BANK_DETAILS_CHANGED [RULE, +30] Invoice bank account ****5570 (Capitec Bank) does not match the verified account on file ****2190 (Absa Bank). POL-001 r
  ▲ EVIDENCE EMAIL_DOMAIN_MISMATCH [RULE, +25] Invoice contact domain "digitalprint-co.co.za" differs from the verified supplier domain "digitalprint.co.za" - a possib
  ▲ EVIDENCE URGENCY_INDICATOR [RULE, +10] Payment is marked IMMEDIATE/urgent. POL-004 treats urgency as a social-engineering signal requiring extra scrutiny.
  ▲ EVIDENCE SOCIAL_ENGINEERING [AI, +0] The invoice uses a classic pretext of an account being temporarily frozen due to an ongoing audit to direct payments to 
  ▲ EVIDENCE SOCIAL_ENGINEERING [AI, +20] The supplier attempts to discourage phone verification by instructing not to call their office line. (Invoice says: "Ple
  ■ R

  ► RECOMMENDATION HOLD_PAYMENT: Keep the payment on hold. Finance must confirm the new bank account by phone AND email, using the contact details in the original onboarding records (never those on the invoice), and then mark the supplier verified (POL-
  ⏸ PAUSED for a human decision. Options: APPROVE_PAYMENT, REQUEST_VERIFICATION, ESCALATE, HOLD_PAYMENT, REJECT_INVOICE


## 4. The paused run lives in Postgres

The checkpointer saved the state after every step (`durability="sync"`). The run is waiting at `human_decision`.
It could wait days (for a second approver or for supplier verification) and survive a restart.

In [5]:
config = service._config(case_id, 1)
snapshot = service.graph.get_state(config)
print("thread:", config["configurable"]["thread_id"])
print("next  :", snapshot.next)
print("interrupt payload:", snapshot.interrupts[0].value)
print("score :", snapshot.values["risk"]["score"], snapshot.values["risk"]["level"])

thread: CASE-1788FFAB:run-1
next  : ('human_decision',)
interrupt payload: {'investigation_id': 'CASE-1788FFAB', 'recommended_action': 'HOLD_PAYMENT', 'options': ['APPROVE_PAYMENT', 'REQUEST_VERIFICATION', 'ESCALATE', 'HOLD_PAYMENT', 'REJECT_INVOICE'], 'last_action_result': None}
score : 85 CRITICAL


## 5. Policy in action: a changed bank account can't be paid

ADR-029: approval is refused until finance has verified the new account by **phone and email**, using contacts
from the onboarding records. A person can place the payment on **hold**, and the case stays open (ADR-030).

In [6]:
for action, who in [(HumanAction.APPROVE_PAYMENT, sipho), (HumanAction.HOLD_PAYMENT, thandi)]:
    r = service.act(case_id, action, who)
    print(f"{who.name:15s} {action.value:16s} → ok={r.ok} ({r.status_code}) {r.message}")

Sipho Dlamini   APPROVE_PAYMENT  → ok=False (409) The bank account on this invoice differs from the verified record. Payment stays on hold until the finance team verifies the new account by phone and email and marks the supplier as verified.


Thandi Nkosi    HOLD_PAYMENT     → ok=True (200) Payment placed on hold pending verification.


## 6. Supplier verification: phone AND email, from trusted sources

Three attempts: (a) using the phone number printed **on the invoice** is refused outright; (b) the supplier
confirms its **old** Absa account, so the case must **not** clear; (c) the supplier confirms the new account.

In [7]:
from trustagent.workflow.verification import VerificationEvidence, VerificationRejected, apply_supplier_verification


def evidence(account, phone_source="onboarding_record"):
    return VerificationEvidence(
        phone_confirmed=True,
        phone_contact="+27 12 444 8820",
        phone_source=phone_source,
        email_confirmed=True,
        email_contact="invoices@digitalprint.co.za",
        email_source="onboarding_record",
        confirmed_bank_account=account,
    )


try:
    with sf() as s, s.begin():
        apply_supplier_verification(s, "SUP-003", evidence("18450029375570", phone_source="invoice"), sipho, None)
except VerificationRejected as e:
    print("(a) REFUSED:", e)

with sf() as s, s.begin():
    out = apply_supplier_verification(s, "SUP-003", evidence("****2190"), sipho, None)
print("(b) old account confirmed → cases cleared:", out.cases_verified, "| still on hold:", out.cases_mismatched)

with sf() as s, s.begin():
    out = apply_supplier_verification(s, "SUP-003", evidence("18450029375570"), sipho, None)
print("(c) new account confirmed → cases cleared:", out.cases_verified)

(a) REFUSED: Cannot mark the supplier verified: the phone number and email address must come from the original onboarding records (or, for a new supplier, another independent source such as the company register), never from the invoice.
(b) old account confirmed → cases cleared: [] | still on hold: ['CASE-1788FFAB']
(c) new account confirmed → cases cleared: ['CASE-1788FFAB']


## 7. Approving: three controls (ADR-048)

1. **Re-run first.** The supplier was verified *after* this assessment, so the score must be refreshed before
   anyone approves.
2. **Separation of duties.** Sipho verified the new bank account, so Sipho can't approve the payment.
3. **Risk-based authority.** A HIGH or CRITICAL case needs a Finance Manager or Department Head to approve.

After the re-run the invoice's account matches the verified record, so the bank-change finding is gone. Lerato
(Department Head) approves, and the run ends.

In [8]:
r = service.act(case_id, HumanAction.APPROVE_PAYMENT, lerato)
print(f"1. Lerato approves before a re-run  → ok={r.ok}: {r.message}")

show(service.rerun(case_id, thandi))
config = service._config(case_id, 2)  # the re-run is a new thread

r = service.act(case_id, HumanAction.APPROVE_PAYMENT, sipho)
print(f"2. Sipho (who verified) approves → ok={r.ok}: {r.message}")
r = service.act(case_id, HumanAction.APPROVE_PAYMENT, lerato)
print(f"3. Lerato approves               → ok={r.ok} closed={r.closed}: {r.message}")
print("next:", service.graph.get_state(config).next or "(finished)")

1. Lerato approves before a re-run  → ok=False: The supplier was verified after this assessment. Re-run the investigation so the risk reflects the verified details, then approve.


  · Invoice and supplier loaded: INV-2003 from Digital Print Co (Pty) Ltd
  · Rule checks completed: 2 finding(s), 2 check(s) passed


  · Policies and contracts retrieved: contract on file; 8 passage(s): DPC-2025-009 (contract), FIN-POL-01 (policy)


  · AI document review completed: 3 observation(s)
  · Contract check completed: 0 price deviation(s), 3 line(s) at the contract rate


  · Deep dive started: Risk is MEDIUM: the outcome could go either way.


  · Deep dive completed: Tools used: supplier_payment_history, similar_invoices, open_cases_for_supplier. I reviewed the payment history, similar invoices, and open cases for 
  ▲ EVIDENCE EMAIL_DOMAIN_MISMATCH [RULE, +25] Invoice contact domain "digitalprint-co.co.za" differs from the verified supplier domain "digitalprint.co.za" - a possib
  ▲ EVIDENCE URGENCY_INDICATOR [RULE, +10] Payment is marked IMMEDIATE/urgent. POL-004 treats urgency as a social-engineering signal requiring extra scrutiny.
  ▲ EVIDENCE SOCIAL_ENGINEERING [AI, +20] The invoice falsely claims the account is frozen due to an audit and demands payment to a new account. (Invoice says: "D
  ▲ EVIDENCE SOCIAL_ENGINEERING [AI, +0] The supplier attempts to discourage telephone verification by telling the client not to call their office line. (Invoice
  ▲ EVIDENCE SOCIAL_ENGINEERING [AI, +0] Urgent wording and threat of withholding the print job are used to pressure immediate payment. (Invoice says: "Payment r
  ■ RISK 5

  ► RECOMMENDATION REQUEST_VERIFICATION: Confirm with the supplier, through its known contacts, that the account holder and contact email on the invoice are genuinely theirs.
  ⏸ PAUSED for a human decision. Options: APPROVE_PAYMENT, REQUEST_VERIFICATION, ESCALATE, HOLD_PAYMENT, REJECT_INVOICE


2. Sipho (who verified) approves → ok=False: Sipho Dlamini verified this supplier's bank details, so someone else must approve the payment (separation of duties).


3. Lerato approves               → ok=True closed=True: Payment approved and released.
next: (finished)


## 8. Checkpoint history (run 2)

Every step is a checkpoint. `interrupt()` shows up as the run pausing at `human_decision`; each resume adds new
checkpoints on the same thread. Run 1's thread was deleted at re-run so it can never be resumed; the audit log
below still covers both runs.

In [9]:
history = list(service.graph.get_state_history(config))
pd.DataFrame(
    [
        {
            "step": h.metadata.get("step"),
            "source": h.metadata.get("source"),
            "next node": ", ".join(h.next) or "(finished)",
        }
        for h in reversed(history)
    ]
)

,step,source,next node
0,-1,input,__start__
1,0,loop,load_context
2,1,loop,rule_checks
3,2,loop,retrieve_context
4,3,loop,ai_review
5,4,loop,contract_check
6,5,loop,provisional_score
7,6,loop,deep_dive
8,7,loop,score
9,8,loop,report


## 9. The audit trail (append-only: the database refuses UPDATE and DELETE)

In [10]:
with sf() as s:
    rows = s.scalars(select(AuditLogRow).where(AuditLogRow.investigation_id == case_id).order_by(AuditLogRow.id)).all()
audit = pd.DataFrame([{"actor": r.actor, "action": r.action, "detail": r.detail[:120]} for r in rows])
audit

,actor,action,detail
0,Thandi Nkosi,Invoice uploaded,"INV-2003-digital-print-co-BANK-CHANGE.pdf (pdf, 1 page(s)); fields read by AI extraction."
1,system,Supplier matched,"'Digital Print Co (Pty) Ltd' matched SUP-003 (Digital Print Co), similarity 1.00."
2,Thandi Nkosi (Finance Analyst),Investigation requested,
3,system,Investigation run 1 started,"Invoice INV-2003 (DOC-757737E5), supplier SUP-003."
4,system,Risk assessed: 85/100 (CRITICAL),5 finding(s) counted; each type counts once.
5,agent,Investigation report generated,Recommended HOLD_PAYMENT.
6,Sipho Dlamini (Finance Manager),Action refused: APPROVE_PAYMENT,The bank account on this invoice differs from the verified record. Payment stays on hold until the finance...
7,Thandi Nkosi (Finance Analyst),Payment placed on hold,Held by Thandi Nkosi (Finance Analyst). Payment for invoice INV-2003 is on hold. The bank account differs ...
8,Sipho Dlamini (Finance Manager),Supplier verified - bank details still differ,"Digital Print Co was verified with account ****2190, but this invoice pays ****5570. Verification NOT appl..."
9,Sipho Dlamini (Finance Manager),Supplier verified,"Account ****5570 confirmed by phone (+27 12 444 8820) and email (invoices@digitalprint.co.za), both from t..."


## 10. Dual authorisation across two resumes (POL-002)

INV-1051 (Secure IT Solutions, R106,950, JSON, so no extraction call). Over R100,000 needs a Finance Manager
**and** a Department Head, two different people. A Finance Analyst can't approve it.

In [11]:
doc = settings.invoices_dir / "INV-1051-secure-it-solutions.json"
with sf() as s, s.begin():
    case2 = intake_document(s, doc.name, doc.read_bytes(), submitted_by="Thandi Nkosi").investigation_id
show(service.start(case2, thandi))
print()
for who in (thandi, sipho, sipho, lerato):
    r = service.act(case2, HumanAction.APPROVE_PAYMENT, who)
    print(f"{who.name:15s} ({who.role.value:16s}) → ok={r.ok} closed={r.closed}: {r.message}")

  · Invoice and supplier loaded: INV-1051 from Secure IT Solutions
  · Rule checks completed: 2 finding(s), 1 check(s) passed


  · Policies and contracts retrieved: no contract on file; 4 passage(s): FIN-POL-01 (policy)


  · AI document review completed: 0 observation(s)


  · Deep dive started: Risk is MEDIUM: the outcome could go either way.


  · Deep dive completed: Tools used: supplier_payment_history, similar_invoices, open_cases_for_supplier. I investigated the supplier invoice using the available tools, includ
  ▲ EVIDENCE SUPPLIER_NOT_VERIFIED [RULE, +15] Secure IT Solutions is not a verified supplier, so there is no confirmed banking record to compare against. Treat as new
  ▲ EVIDENCE AMOUNT_EXCEEDS_THRESHOLD [RULE, +15] Amount R106,950 exceeds the R100,000 threshold with no supplier-specific range on file. POL-002 requires dual authorizat
  ■ RISK 30/100 MEDIUM


  ► RECOMMENDATION REQUEST_VERIFICATION: Complete new-supplier onboarding: confirm the company, its bank account and its contacts by phone and email using an independent source such as the company register, never the invoice. POL-002: approval needs both a Fina
  ⏸ PAUSED for a human decision. Options: APPROVE_PAYMENT, REQUEST_VERIFICATION, ESCALATE, HOLD_PAYMENT, REJECT_INVOICE



Thandi Nkosi    (FINANCE_ANALYST ) → ok=False closed=False: Payments over R100,000 need approval from a Finance Manager and a Department Head (POL-002). A Finance Analyst cannot approve this payment.


Sipho Dlamini   (FINANCE_MANAGER ) → ok=True closed=False: Approval recorded. Awaiting Department Head.


Sipho Dlamini   (FINANCE_MANAGER ) → ok=False closed=False: Sipho Dlamini has already approved this payment. A second approval must come from a different person.


Lerato Khumalo  (DEPARTMENT_HEAD ) → ok=True closed=True: Payment approved and released.


## 11. Graceful degradation (simulated outage, clearly labelled)

Here the AI steps are *replaced by functions that fail*, the way a quota error or network failure would.
(During the build, the real Gemini free tier hit its 15 requests/min limit mid-run, and exactly this path handled it.)
The investigation still completes: rule-based score, a cautious recommendation from the minimum-action table, and
a clear note that the AI review was unavailable.

In [12]:
from trustagent.graph.nodes import Deps
from trustagent.llm.factory import LLMUnavailable


def outage(_):
    raise LLMUnavailable("simulated outage: 429 RESOURCE_EXHAUSTED")


down = InvestigationService.create(
    settings.test_database_url, deps=Deps(session_factory=sf, reviewer=outage, reporter=outage, deep_diver=outage)
)
doc = settings.invoices_dir / "INV-1048-abc-office-solutions.json"
with sf() as s, s.begin():
    case3 = intake_document(s, doc.name, doc.read_bytes(), submitted_by="Thandi Nkosi").investigation_id
show(down.start(case3, thandi))
down.close()

  · Invoice and supplier loaded: INV-1048 from ABC Office Solutions
  · Rule checks completed: 3 finding(s), 2 check(s) passed
  · Policies and contracts retrieved: contract n/a; 4 passage(s): POL-001 (policy), POL-002 (policy), POL-003 (policy), POL-004 (policy)


  · AI review unavailable: Continuing with rule checks only.


  · Deep dive started: Score 60 is within 5 points of the 60 boundary.
  · Deep dive unavailable: Continuing without it.
  ▲ EVIDENCE BANK_DETAILS_CHANGED [RULE, +30] Invoice bank account ****9917 (Capitec Bank) does not match the verified account on file ****4821 (First National Bank).
  ▲ EVIDENCE UNUSUAL_AMOUNT [RULE, +20] Amount R185,000 exceeds this supplier's expected range of R15,000-R40,000.
  ▲ EVIDENCE URGENCY_INDICATOR [RULE, +10] Payment is marked IMMEDIATE/urgent. POL-004 treats urgency as a social-engineering signal requiring extra scrutiny.
  ■ RISK 60/100 HIGH


  ► RECOMMENDATION HOLD_PAYMENT: Risk level is HIGH. The bank account differs from the verified record (POL-001). Payment must stay on hold until the finance team confirms the new account by phone AND email, using contact details from the original onboa
  ⏸ PAUSED for a human decision. Options: APPROVE_PAYMENT, REQUEST_VERIFICATION, ESCALATE, HOLD_PAYMENT, REJECT_INVOICE


## 12. Cost accounting

Every model call's tokens are recorded on the case (`investigations.llm_usage`); the evaluation turns these
into cost per investigation.

In [13]:
with sf() as s:
    usage = s.scalar(select(InvestigationRow.llm_usage).where(InvestigationRow.id == case_id))
pd.DataFrame(usage).groupby("step")[["input_tokens", "output_tokens"]].sum()

,input_tokens,output_tokens
step,,
ai_review,2561,731
deep_dive,1747,95
report,1020,205


In [14]:
service.close()